In [1]:
using SBMLToolkit, ModelingToolkit, DifferentialEquations, StochasticDiffEq
using Plots
using DataFrames
using CSV
using Random
using Distributions
using SBML
using SymbolicUtils
using StaticArrays
using Catalyst
using AdvancedMH
using MCMCChains
using MCMCChainsStorage
using AbstractMCMC
using StatsPlots
using HDF5
using NPZ

using Particles
using ParticlesDE
using StaticDistributions

base_dir = dirname(dirname(pwd()))

include(joinpath(base_dir, "code/utils/utilities.jl"))
include(joinpath(base_dir, "code/epmodels/utils/posEM.jl"))

## SIS-model filter
In this notebook we evaluate the PMMH algorithm for the simple SIS-model and investigate the results.

In [ ]:
# import the model simulator
model_file = joinpath(base_dir, "code/epmodels/sis_model.jl")
include(model_file)

SIR_Model_Ensemble_Simulation (generic function with 1 method)

In [ ]:
# SIR-model settings and parameters
N = 180000
init_I = 1800
init_S = N - init_I
u0 = [init_S/N; init_I/N]

endtime = 100.0
tspan = (0.0, endtime)

# define SDe problem
SDE_problem = SIS_SDEProblem(nothing, N, endtime=endtime, initial_state=u0);

In [5]:
# simulate sde problem
solve_kwargs = (dt=1e-2, dense=true, force_dtmin=true)

sde_sol = solve(SDE_problem, PositiveEM(); solve_kwargs...);

## Evaluation of inference results

In [7]:
include(joinpath(base_dir, "code/utils/EvaluateParticleFilter.jl"))

MCMC_diagnostics (generic function with 1 method)

### Synthetic data

In [34]:
# filter settings
nparticles=100
nchains = 4
niter = 50000

# set noise model
noise_model ="normal"

# set experiment id
ex_id = 1

#set prior
prior = "normal";

In [35]:
# set dataset 
datasets = ["1", "2"]
dataset = datasets[ex_id]

# SIR-model settings and parameters
N = 180000
init_I = 1800
init_S = N - init_I
u0 = [init_S/N; init_I/N]

endtime = 100.0
tspan = (0.0, endtime)

# define SDe problem
SDE_problem = SIR_SDEProblem(nothing, N, endtime=endtime, initial_state=u0);

# parameters are in the ordering [beta, gamma]
true_pars = [
    [0.1, 0.05], 
    [0.22, 0.2]]
true_par = true_pars[parse(Int, dataset)]

# get true parameter dictionary
true_par_dict = Dict(
    :beta => true_par[1],
    :gamma => true_par[2];)

Dict{Symbol, Float64} with 2 entries:
  :beta  => 0.1
  :gamma => 0.05

In [ ]:
# load results
chain = h5open(joinpath(base_dir, "output/PF_Experiments/SIS/sis_$(dataset)/SIS_$(dataset)_$(nchains)chs_$(niter)it_$(nparticles)p.h5"), "r") do f
    read(f, Chains)
 end
chain_time = open(joinpath(base_dir, "output/PF_Experiments/SIS/sis_$(dataset)/time_SIS_$(dataset)_$(nchains)chs_$(niter)it_$(nparticles)p.txt"), "r") do f
    read(f, Float64)
end
chain = setinfo(chain, (start_time=0.0, stop_time=chain_time));

if niter > 10000
    burnin = Int(niter-10000)
else
    burnin = Int(niter/10)
end
    
mixed_chain = chain[burnin:end]

# check which chain is converged and remove stuck chains.
nparams = length(names(mixed_chain))
stuck_flags = Bool[]
for c in 1:nchains
    chain_slice = mixed_chain[:, :, c]
    n_samples = size(chain_slice, 1)
    last_samples = chain_slice[:, :, 1]
    # Collect quantiles for each parameter
    q_by_param = [quantile(vec(last_samples[:, param, :]), [0.05, 0.5, 0.95]) for param in names(mixed_chain)[2:end]]
    # Check if all quantiles are equal across parameters (i.e. chain not moving)
    all_same = all(q -> all(abs.(q[1] .- q[2:end]) .< 1e-10), q_by_param)
    push!(stuck_flags, all_same)
end
mixed_chain = mixed_chain[:,:, findall(!, stuck_flags)]

if size(mixed_chain, 3) == 0
   error("All chains got stuck! No usable chains remain.")
end
mixed_chain

In [ ]:
n_samples = 10000
df = DataFrame(mixed_chain)
if nrow(df) >= n_samples
    idx = randperm(nrow(df))[1:n_samples]
    posterior_df = df[idx, :]
else 
    posterior_df = df
end
param_cols = names(posterior_df, Not([:chain, :iteration, :lp]))
parameter_df = posterior_df[:, param_cols]
parameter_df[:, :gamma] .= 1 ./parameter_df[:, :gamma]

CSV.write(joinpath(base_dir, "./output/PF_samples/SIS/pf_posterior_sis_$(dataset).csv"), parameter_df)

In [ ]:
# optionally visualize marginal densities and store them
# visualize_chain(mixed_chain, true_par_dict=true_par_dict, save_path=nothing)

# create diagnostics to assess convergence and exploration based on the last 10.000 samples of non-stuck chains extracted above
diagnostics_df = MCMC_diagnostics(mixed_chain[], autocorlag=250);

# store diagnostics DataFrame
# CSV.write(joinpath(base_dir, "output/PF_Experiments/SIS/sis_$(dataset)_diagnostics.csv"), diagnostics_df)
